In [1]:
import pandas as pd
import yfinance as yf
import ta
import pandas_datareader.data as web
import numpy as np
import torch
import quantstats as qs
import os
import warnings

# ปิด Warning ที่ไม่จำเป็นเพื่อความสะอาดของหน้าจอตอนรัน
warnings.filterwarnings('ignore')

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback

In [2]:
# ==========================================
# 0. HARDWARE SETUP
# ==========================================
device = "cpu" #DRL เหมาะกับ CPU มากกว่า
print(f"✅ Device Strategy: '{device}' เพราะ DRL เหมาะกับ CPU มากกว่า")
if torch.cuda.is_available():
    print(f"   - พบ GPU: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)")
else:
    print("   - ไม่พบการ์ดจอแยก ระบบจะทำงานบน CPU (ซึ่งเหมาะกับโปรเจกต์นี้และทำงานได้รวดเร็วผ่าน Multi-core)")

✅ Device Strategy: 'cpu' เพราะ DRL เหมาะกับ CPU มากกว่า
   - พบ GPU: NVIDIA GeForce RTX 5060 Ti (VRAM: 17.10 GB)


In [3]:
# ==========================================
# 1. GLOBAL CONFIGURATION
# ==========================================
TICKERS = ['DIA','QQQ','SPY']
TECHNICAL_INDICATORS = ['MACD_12_26_9', 'RSI_14', 'EMA_50', 'STOCHRSIk_14_14_3_3']
MACRO_INDICATORS = ['vix', 'bond_yield', 'gold', 'wti', 'fed_rate', 'm2']
FEATURES = TECHNICAL_INDICATORS + MACRO_INDICATORS

In [4]:
# ==========================================
# 2. HIGH-PERFORMANCE DATA PIPELINE
# ==========================================
def fetch_and_prepare_data(tickers, start_date, end_date):
    print(f"📦 กําลังดาวน์โหลดข้อมูลจาก {start_date} ถึง {end_date}...")
    processed_dfs = []
    
    for ticker in tickers:
        try:
            df = yf.download(ticker, start=start_date, end=end_date, progress=False)
            if df.empty: continue
            
            # จัดการ Multi-index Columns จาก yfinance เวอร์ชันใหม่
            df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
            df = df.reset_index()
            df.rename(columns={'Date': 'date', 'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Volume': 'volume'}, inplace=True)
            df['tic'] = ticker
            
            # คำนวณ Technical Indicators
            df['MACD_12_26_9'] = ta.trend.MACD(close=df['close']).macd()
            df['RSI_14'] = ta.momentum.RSIIndicator(close=df['close'], window=14).rsi()
            df['EMA_50'] = ta.trend.EMAIndicator(close=df['close'], window=50).ema_indicator()
            df['STOCHRSIk_14_14_3_3'] = ta.momentum.StochRSIIndicator(close=df['close'], window=14).stochrsi_k()
            
            processed_dfs.append(df)
        except Exception as e:
            print(f"❌ ไม่สามารถดาวน์โหลด {ticker}: {e}")
            
    if not processed_dfs:
        raise ValueError("ดาวน์โหลดข้อมูลล้มเหลว กรุณาตรวจสอบการเชื่อมต่ออินเทอร์เน็ต")
        
    final_df = pd.concat(processed_dfs, ignore_index=True)
    
    # ดึงข้อมูลสินค้าโภคภัณฑ์และดัชนีความกลัว (VIX)
    macro_tickers = {"^VIX": "vix", "^TNX": "bond_yield", "GC=F": "gold", "CL=F": "wti"}
    try:
        macro_raw = yf.download(list(macro_tickers.keys()), start=start_date, end=end_date, progress=False)
        macro_df = macro_raw['Close'].copy()
        macro_df.columns = [col[0] if isinstance(col, tuple) else col for col in macro_df.columns]
        macro_df.rename(columns=macro_tickers, inplace=True)
        macro_df = macro_df.reset_index().rename(columns={'Date': 'date'})
        final_df = pd.merge(final_df, macro_df, on='date', how='left')
    except Exception:
        for name in macro_tickers.values(): final_df[name] = 0.0

    # ดึงข้อมูล FRED (อัตราดอกเบี้ยนโยบายและปริมาณเงินในระบบ)
    try:
        fed_funds = web.DataReader('FEDFUNDS', 'fred', start_date, end_date)
        m2_supply = web.DataReader('M2SL', 'fred', start_date, end_date)
        macro_fred = pd.merge(fed_funds, m2_supply, left_index=True, right_index=True, how='outer')
        macro_fred.rename(columns={'FEDFUNDS': 'fed_rate', 'M2SL': 'm2'}, inplace=True)
        macro_fred = macro_fred.reset_index().rename(columns={'DATE': 'date'})
        final_df = pd.merge(final_df, macro_fred, on='date', how='left')
    except Exception:
        final_df['fed_rate'] = 0.0
        final_df['m2'] = 0.0
    
    # จัดเรียง อุดช่องโหว่ข้อมูล และแปลงวันที่ให้เป็นดัชนี (Day)
    final_df.sort_values(['date', 'tic'], inplace=True)
    final_df = final_df.ffill().bfill()
    
    date_list = sorted(final_df['date'].unique())
    date2day = {date: day for day, date in enumerate(date_list)}
    final_df['day'] = final_df['date'].map(date2day)
    final_df['date'] = final_df['date'].dt.strftime('%Y-%m-%d')
    
    final_df = final_df.sort_values(['date', 'tic'])
    final_df.index = final_df['day'].values
    return final_df

In [5]:
# ==========================================
# 3. ROBUST CUSTOM ENVIRONMENT
# ==========================================
class RealisticTradingEnv(StockTradingEnv):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # ⚡️ แคชข้อมูล VIX ลง Numpy Array ล่วงหน้า เพื่อความรวดเร็วในการเทรน
        self.vix_array = self.df['vix'].values if 'vix' in self.df.columns else np.zeros(len(self.df))
        
    def _calculate_reward(self):
        current_portfolio_value = self.state[0] + sum(
            np.array(self.state[1:(self.stock_dim + 1)]) *
            np.array(self.state[(self.stock_dim + 1):(self.stock_dim * 2 + 1)])
        )
        
        previous_portfolio_value = self.asset_memory[-1] if len(self.asset_memory) > 0 else self.initial_amount
        if previous_portfolio_value <= 0:
            previous_portfolio_value = self.initial_amount
            
        step_return = (current_portfolio_value - previous_portfolio_value) / previous_portfolio_value
        transaction_costs = self.cost * self.reward_scaling
        
        # 🔥 1. Asymmetric Reward (ทำโทษการขาดทุน 2 เท่าเพื่อสร้างพฤติกรรมกลัวการขาดทุน)
        if step_return < 0:
            step_return *= 2.0  
            
        # ⚡️ 2. ดึงค่า VIX ด้วย Numpy Array ตรงๆ (เร็วขึ้นกว่าเดิมหลายสิบเท่า)
        current_vix = self.vix_array[self.day] if self.day < len(self.vix_array) else 0.0
        invested_ratio = 1 - (self.state[0] / current_portfolio_value) if current_portfolio_value > 0 else 0
        
        # 🔥 3. Continuous Market Risk Penalty (ทำโทษถ้ารับความเสี่ยงมากไปในภาวะตลาดผันผวน)
        risk_penalty = 0.0
        if current_vix > 20.0:
            vix_excess = (current_vix - 20.0) / 100.0 
            risk_penalty = vix_excess * invested_ratio * (abs(step_return) + 0.001)
            
        # 🔥 4. ปรับสเกลให้อยู่ในย่านที่ PPO เรียนรู้ได้ดีสุด และ Clip ค่าป้องกันปัญหา Black Swan
        reward = (step_return * 100) - transaction_costs - (risk_penalty * 100)
        reward = np.clip(reward, -10.0, 10.0)
        
        return float(reward)

def get_env_kwargs(stock_dimension):
    state_space = 1 + (2 * stock_dimension) + (len(FEATURES) * stock_dimension)
    return {
        "hmax": 100,
        "initial_amount": 1000000,
        "num_stock_shares": [0] * stock_dimension,
        "state_space": state_space,
        "stock_dim": stock_dimension,
        "tech_indicator_list": FEATURES,
        "action_space": stock_dimension,
        "reward_scaling": 1e-4,
        "buy_cost_pct": [0.001] * stock_dimension,
        "sell_cost_pct": [0.001] * stock_dimension,
    }

In [6]:
# ==========================================
# 4. TRANSLATION & REPORT ENGINE
# ==========================================
def explain_bot_performance(returns):
    print("\n" + "="*55)
    print("🤖 สรุปผลงานบอทเทรดในหน้าต่างทดสอบจริง (ฉบับเข้าใจง่าย)")
    print("="*55)
    
    total_return = qs.stats.comp(returns) * 100
    cagr = qs.stats.cagr(returns) * 100
    max_dd = qs.stats.max_drawdown(returns) * 100
    sharpe = qs.stats.sharpe(returns)
    win_rate = qs.stats.win_rate(returns) * 100

    print("\n💰 1. ด้านการทำกำไร (Return)")
    print(f"   • กำไรสะสมตลอดการทดสอบ: {total_return:.2f}%")
    print(f"   • ผลตอบแทนทบต้นเฉลี่ยรายปี (CAGR): {cagr:.2f}%")
    if cagr > 12:    print("   ✅ สรุป: ยอดเยี่ยม! บอทสร้างผลงานชนะอัตราเฉลี่ยของตลาดส่วนใหญ่")
    elif cagr > 0:  print("   ⚠️ สรุป: พอใช้ได้ พอร์ตโตขึ้นแต่ยังไม่โดดเด่นนัก")
    else:            print("   ❌ สรุป: ล้มเหลว บอททำเงินต้นสูญหาย")

    print("\n📉 2. ด้านความเสี่ยง (Risk & Drawdown)")
    print(f"   • ช่วงที่พอร์ตร่วงหนักสุดจากจุดสูงสุด (Max Drawdown): {max_dd:.2f}%")
    if max_dd > -15:  print("   ✅ สรุป: ปลอดภัยสูง ระบบคุมสัดส่วนการสูญเสียเงินได้ดีมาก")
    elif max_dd > -30: print("   ⚠️ สรุป: ปานกลาง พอร์ตแกว่งตามมาตรฐานสไตล์กองทุนเชิงรุก")
    else:             print("   ❌ สรุป: อันตรายมาก! บอทปล่อยให้พอร์ตเสียหายหนักเกินไป")

    print("\n⚖️ 3. ความคุ้มค่าและสถิติการชนะ (Efficiency)")
    print(f"   • ความคุ้มค่าต่อหนึ่งหน่วยความเสี่ยง (Sharpe Ratio): {sharpe:.2f}")
    if sharpe >= 1.0: print("   ✅ สรุป: ดีเยี่ยม กำไรที่ได้คุ้มค่าอย่างมากกับความเสี่ยงที่ถือครอง")
    else:             print("   ⚠️ สรุป: ความคุ้มค่าน้อยลง ผลตอบแทนอาจไม่สมน้ำสมเนื้อกับความเสี่ยงที่เผชิญ")
    print(f"   • อัตราความแม่นยำรายวัน (Win Rate): {win_rate:.2f}%")
    print("="*55 + "\n")

    START_DATE = '1999-03-01'
    END_DATE = '2006-12-31'


    TEST_START_DATE = '2007-01-01'
    TEST_END_DATE = '2026-07-29'

In [7]:
# ==========================================
# 5. HIGH-SPEED TRAINING PIPELINE WITH VALIDATION
# ==========================================
def train_model():
    # ⏳ แบ่งเวลา 3 ชุดอิสระป้องกัน Data Leakage
    TRAIN_START, TRAIN_END = '1999-04-01', '2019-12-31'
    VAL_START, VAL_END     = '2020-01-01', '2023-12-31'
    
    # สร้างโฟลเดอร์สำหรับ Export CSV
    os.makedirs("./exported_data", exist_ok=True)
    
    print("\n--- 🛠️ [1/3] เริ่มเตรียมข้อมูลสําหรับฝึกสอนบอท (Training Set) ---")
    train_df = fetch_and_prepare_data(TICKERS, TRAIN_START, TRAIN_END)
    
    train_df.to_csv("./exported_data/train_set.csv", index=False)
    print("📁 [Exported] บันทึกไฟล์ Training Set เรียบร้อย -> ./exported_data/train_set.csv")
    
    env_kwargs = get_env_kwargs(len(TICKERS))
    
    # ⚡️ ฟังก์ชันสร้าง Env สำหรับ Multi-core
    def make_env():
        return lambda: RealisticTradingEnv(df=train_df, **env_kwargs)
        
    NUM_CPU = 4  
    env_train = SubprocVecEnv([make_env() for _ in range(NUM_CPU)])
    
    print("\n--- 🛠️ [2/3] เริ่มเตรียมข้อมูลสําหรับข้อสอบ (Validation Set) ---")
    val_df = fetch_and_prepare_data(TICKERS, VAL_START, VAL_END)
    
    val_df.to_csv("./exported_data/validation_set.csv", index=False)
    print("📁 [Exported] บันทึกไฟล์ Validation Set เรียบร้อย -> ./exported_data/validation_set.csv")
    
    # ใช้ DummyVecEnv สำหรับ Validation (เพื่อไม่ให้กินทรัพยากรซ้ำซ้อน)
    env_val = DummyVecEnv([lambda: RealisticTradingEnv(df=val_df, **env_kwargs)])
    
    # 🎯 ตั้งค่า EvalCallback แจ้งเตือนเซฟโมเดลที่ดีที่สุด
    os.makedirs("./best_model", exist_ok=True)
    eval_callback = EvalCallback(
        env_val, 
        best_model_save_path='./best_model/',
        log_path='./best_model/logs/',
        eval_freq=max(500, 2048 // NUM_CPU), 
        deterministic=True,
        render=False
    )
    
    agent = PPO(
        "MlpPolicy",
        env_train,
        learning_rate=0.00025,
        n_steps=2048,
        batch_size=256, 
        device=device,
        verbose=1
    )
    
    print("\n--- 🚀 [3/3] บอทเริ่มกระโจนเข้าสู่การเรียนรู้แบบคู่ขนาน ---")
    try:
        # เทรนทั้งหมด 15,000 สเตป (เพิ่มขึ้นเล็กน้อยเพื่อให้โมเดลมีโอกาสเรียนรู้เต็มที่)
        agent.learn(total_timesteps=15000, callback=eval_callback) 
        agent.save("ppo_realistic_trading_bot_last")
        print("💾 บันทึกโมเดลเสร็จสิ้น! บอทเวอร์ชันที่ดีที่สุดถูกบรรจุอยู่ในโฟลเดอร์ './best_model/'")
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดในจังหวะเทรนโมเดล: {e}")

In [8]:
# ==========================================
# 6. COMPREHENSIVE OUT-OF-SAMPLE TEST
# ==========================================
def test_model():
    # ⏳ อัปเดตช่วงเวลาสำหรับ Test Set ให้เป็นช่วงปัจจุบันที่สุดถึงปัจจุบันปี 2026
    TEST_START, TEST_END = '2024-01-01', '2026-07-29'
    
    print("\n🔮 ดึงข้อมูลตลาดปัจจุบันมาสับไพ่ทดสอบจริง (Out-of-Sample Test)...")
    test_df = fetch_and_prepare_data(TICKERS, TEST_START, TEST_END)
    
    os.makedirs("./exported_data", exist_ok=True)
    test_df.to_csv("./exported_data/test_set.csv", index=False)
    print("📁 [Exported] บันทึกไฟล์ Test Set เรียบร้อย -> ./exported_data/test_set.csv")
    
    # 🎯 โหลดโมเดลตัวที่ทำคะแนน Validation ได้สูงที่สุดมาใช้
    model_path = "./best_model/best_model.zip"
    if not os.path.exists(model_path):
        # ลบ .zip ออก เพราะ Stable-Baselines3 จะต่อท้ายให้เองหากหาไฟล์ดิบไม่เจอ
        model_path = "ppo_realistic_trading_bot_last" 
        print("⚠️ ไม่พบโมเดลคัดสรรพิเศษยามทำลายสถิติ วนกลับไปใช้โมเดลรอบสุดท้าย")
        
    trained_model = PPO.load(model_path, device=device)
    
    env_kwargs = get_env_kwargs(len(TICKERS))
    e_test_gym = RealisticTradingEnv(df=test_df, **env_kwargs)
    env_test = DummyVecEnv([lambda: e_test_gym])
    
    obs = env_test.reset()
    account_memory = []
    num_days = len(test_df.index.unique())
    
    print("📈 บอทกำลังดำเนินการจำลองการกระจายสินทรัพย์จริงในอดีต...")
    for i in range(num_days):
        action, _ = trained_model.predict(obs, deterministic=True)
        obs, rewards, dones, info = env_test.step(action)
        
        # บันทึกข้อมูลพอร์ตในวันก่อนวันสุดท้ายเพื่อกัน array index error
        if i == num_days - 2:
            account_memory = env_test.env_method(method_name="save_asset_memory")[0]
            
        if dones[0]:
            print("🏁 การทดสอบย้อนหลังสิ้นสุดสมบูรณ์!")
            break

    # แปลงผลลัพธ์เป็น DataFrame เพื่อคำนวณสถิติ
    df_account_value = pd.DataFrame(account_memory)
    df_account_value['date'] = pd.to_datetime(df_account_value['date'])
    df_account_value.set_index('date', inplace=True)
    
    df_account_value['daily_return'] = df_account_value['account_value'].pct_change()
    df_account_value.dropna(inplace=True)
    
    explain_bot_performance(df_account_value['daily_return'])

In [9]:
# ==========================================
# 7. EXECUTION GATEWAY
# ==========================================
if __name__ == "__main__":
    train_model()
    test_model()


--- 🛠️ [1/3] เริ่มเตรียมข้อมูลสําหรับฝึกสอนบอท (Training Set) ---
📦 กําลังดาวน์โหลดข้อมูลจาก 1999-04-01 ถึง 2019-12-31...
📁 [Exported] บันทึกไฟล์ Training Set เรียบร้อย -> ./exported_data/train_set.csv

--- 🛠️ [2/3] เริ่มเตรียมข้อมูลสําหรับข้อสอบ (Validation Set) ---
📦 กําลังดาวน์โหลดข้อมูลจาก 2020-01-01 ถึง 2023-12-31...
📁 [Exported] บันทึกไฟล์ Validation Set เรียบร้อย -> ./exported_data/validation_set.csv
Using cpu device

--- 🚀 [3/3] บอทเริ่มกระโจนเข้าสู่การเรียนรู้แบบคู่ขนาน ---
Eval num_timesteps=2048, episode_reward=0.00 +/- 0.00
Episode length: 1006.00 +/- 0.00
---------------------------------
| eval/              |          |
|    mean_ep_length  | 1.01e+03 |
|    mean_reward     | 0        |
| time/              |          |
|    total_timesteps | 2048     |
---------------------------------
New best mean reward!
day: 1005, episode: 10
begin_total_asset: 1000000.00
end_total_asset: 1000000.00
total_reward: 0.00
total_cost: 0.00
total_trades: 0
Eval num_timesteps=4096, episod